Obs: CUAREIM+04CATALAN es una série calculada como pico de nivel en Cuareím Río + 0.4 * pico de nivel en Catalan Grande.

In [74]:
import pandas as pd
import os
import plotly.graph_objects as go

In [82]:
data = pd.read_csv("C:\\Tiago\\1_Cuenca_Cuareim\\Correlacion_niveles\\peaks_cuareim.csv")

In [83]:
# Convert FECHA_SARANDI, FECHA_POLANCO and FECHA_DURAZNO to datetime. recall NaT means Not a Time, which is used for missing values in datetime columns. Also convert to format dd-MM-yyyy hh:mm:s
data['FECHA_CATALAN'] = pd.to_datetime(data['FECHA_CATALAN'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')
data['FECHA_CUAREIM'] = pd.to_datetime(data['FECHA_CUAREIM'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')
data['FECHA_ARTIGAS'] = pd.to_datetime(data['FECHA_ARTIGAS'], errors='coerce').dt.strftime('%d-%m-%Y %H:%M:%S')

In [84]:
# set ID to index
data.set_index('ID', inplace=True)
#data

In [85]:
fmt = '%d-%m-%Y %H:%M:%S'
data['CATALAN_2_ARTIGAS'] = (pd.to_datetime(data['FECHA_ARTIGAS'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_CATALAN'], format=fmt, errors='coerce')).dt.total_seconds() / 3600
data['CUAREIM_2_ARTIGAS'] = (pd.to_datetime(data['FECHA_ARTIGAS'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_CUAREIM'], format=fmt, errors='coerce')).dt.total_seconds() / 3600
data['CATALAN_2_CUAREIM'] = (pd.to_datetime(data['FECHA_CUAREIM'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_CATALAN'], format=fmt, errors='coerce')).dt.total_seconds() / 3600
data['CUAREIM+04CATALAN_2_CUAREIM'] = (pd.to_datetime(data['FECHA_CUAREIM'], format=fmt, errors='coerce') - pd.to_datetime(data['FECHA_CUAREIM+04CATALAN'], format=fmt, errors='coerce')).dt.total_seconds() / 3600



In [86]:
#data

In [87]:
from plotly.subplots import make_subplots
import numpy as np
import ipywidgets as widgets
from IPython.display import display

site_labels = {
    'CATALAN': 'Catalán Grande',
    'CUAREIM': 'Cuareím Río',
    'ARTIGAS': 'Artigas',
    'CUAREIM+04CATALAN': 'Cuareím+04Catalán',
}

x_widget = widgets.Dropdown(options=[(label, code) for code, label in site_labels.items()], value='CUAREIM', description='Eje X:')
y_widget = widgets.Dropdown(options=[(label, code) for code, label in site_labels.items()], value='ARTIGAS', description='Eje Y:')
output = widgets.Output()

def get_transit_time(site_a, site_b):
    # tiempo de tránsito de site_a a site_b, invirtiendo el signo si solo existe la columna inversa
    col_ab = f'{site_a}_2_{site_b}'
    col_ba = f'{site_b}_2_{site_a}'
    if col_ab in data.columns:
        return data[col_ab]
    if col_ba in data.columns:
        return -data[col_ba]
    return None

def plot_sites(*_):
    x_site, y_site = x_widget.value, y_widget.value
    with output:
        output.clear_output(wait=True)
        if x_site == y_site:
            print('Selecciona dos sitios distintos.')
            return

        fig = make_subplots(rows=2, cols=1)
        row1_domain = fig.layout.yaxis.domain
        row2_domain = fig.layout.yaxis2.domain

        mask = data[x_site].notna() & data[y_site].notna()
        x = data[x_site][mask].values
        y = data[y_site][mask].values
        coeffs = np.polyfit(x, y, 1)
        trendline_x = np.linspace(x.min(), x.max(), 100)
        trendline_y = np.polyval(coeffs, trendline_x)

        residuals = y - np.polyval(coeffs, x)
        std = np.std(residuals)

        equation = f'y = {coeffs[0]:.4f}x + {coeffs[1]:.4f}'

        fig.add_trace(
            go.Scatter(x=data[x_site], y=data[y_site], mode='markers', name='Datos observados', legend='legend', showlegend=True),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(x=trendline_x, y=trendline_y, mode='lines', name=f'Tendencia lineal ({equation})', line=dict(color='red'), legend='legend', showlegend=True),
            row=1, col=1
        )
        fig.add_trace(
            go.Scatter(
                x=np.concatenate([trendline_x, trendline_x[::-1]]),
                y=np.concatenate([trendline_y + std, (trendline_y - std)[::-1]]),
                fill='toself', fillcolor='rgba(255,0,0,0.2)', line=dict(color='rgba(255,255,255,0)'),
                name='Banda de incertidumbre (±1 std)', legend='legend', showlegend=True
            ),
            row=1, col=1
        )

        transit = get_transit_time(x_site, y_site)
        if transit is not None:
            fig.add_trace(
                go.Scatter(x=transit, y=data[y_site], mode='markers', name='Tiempo de tránsito', legend='legend2', showlegend=True),
                row=2, col=1
            )
            fig.update_xaxes(title_text=f'Tiempo tránsito {site_labels[x_site]} → {site_labels[y_site]} (horas)', row=2, col=1)
            fig.update_yaxes(title_text=site_labels[y_site], row=2, col=1)

        fig.add_annotation(
            x=trendline_x[10], y=trendline_y[10],
            text=equation,
            showarrow=True, arrowhead=2,
            font=dict(color='red'),
            xref='x', yref='y'
        )

        fig.update_xaxes(title_text=site_labels[x_site], dtick=1, minor=dict(dtick=0.5, showgrid=True), row=1, col=1)
        fig.update_yaxes(title_text=site_labels[y_site], dtick=1, minor=dict(dtick=0.5, showgrid=True), row=1, col=1)

        fig.update_layout(
            width=800, height=1300,
            legend=dict(orientation='h', x=0.5, xanchor='center', y=row1_domain[0] - 0.08, yanchor='top'),
            legend2=dict(orientation='h', x=0.5, xanchor='center', y=row2_domain[0] - 0.12, yanchor='top'),
            margin=dict(b=140),
        )
        fig.show()

x_widget.observe(plot_sites, names='value')
y_widget.observe(plot_sites, names='value')

display(widgets.HBox([x_widget, y_widget]), output)
plot_sites()


Output()

In [88]:
from sklearn.linear_model import LinearRegression

# Modelos individuales
# CUAREIM -> ARTIGAS
mask_cu = data['CUAREIM'].notna() & data['ARTIGAS'].notna()
x_cu = data.loc[mask_cu, 'CUAREIM'].values.reshape(-1, 1)
y_cu = data.loc[mask_cu, 'ARTIGAS'].values
coeffs_cu = np.polyfit(x_cu.flatten(), y_cu, 1)
residuals_cu = y_cu - np.polyval(coeffs_cu, x_cu.flatten())
std_cu = np.std(residuals_cu)

# CATALAN -> ARTIGAS
mask_ca = data['CATALAN'].notna() & data['ARTIGAS'].notna()
x_ca = data.loc[mask_ca, 'CATALAN'].values.reshape(-1, 1)
y_ca = data.loc[mask_ca, 'ARTIGAS'].values
coeffs_ca = np.polyfit(x_ca.flatten(), y_ca, 1)
residuals_ca = y_ca - np.polyval(coeffs_ca, x_ca.flatten())
std_ca = np.std(residuals_ca)

# CUAREIM+04CATALAN -> ARTIGAS
mask_cc = data['CUAREIM+04CATALAN'].notna() & data['ARTIGAS'].notna()
x_cc = data.loc[mask_cc, 'CUAREIM+04CATALAN'].values.reshape(-1, 1)
y_cc = data.loc[mask_cc, 'ARTIGAS'].values
coeffs_cc = np.polyfit(x_cc.flatten(), y_cc, 1)
residuals_cc = y_cc - np.polyval(coeffs_cc, x_cc.flatten())
std_cc = np.std(residuals_cc)

def predecir_artigas(cuareim_val=None, catalan_val=None, n_std=1):
    """
    Predice el nivel máximo en Artigas según los valores disponibles.

    Parámetros:
        cuareim_val: nivel en Cuareím Río (opcional)
        catalan_val: nivel en Catalán Grande (opcional)
        n_std: número de desviaciones estándar para el rango de incertidumbre
    
    Retorna:
        valor_predicho, (rango_min, rango_max), modelo_usado
    """
    if cuareim_val is not None and catalan_val is not None:
        valor_cc = cuareim_val + 0.4 * catalan_val
        valor = np.polyval(coeffs_cc, valor_cc)
        std_used = std_cc
        modelo = 'Cuareím + 0.4·Catalán (regresión lineal)'
    elif cuareim_val is not None:
        # Modelo simple: CUAREIM -> ARTIGAS
        valor = np.polyval(coeffs_cu, cuareim_val)
        std_used = std_cu
        modelo = 'Solo Cuareím Río (regresión simple)'
    elif catalan_val is not None:
        # Modelo simple: CATALAN -> ARTIGAS
        valor = np.polyval(coeffs_ca, catalan_val)
        std_used = std_ca
        modelo = 'Solo Catalán Grande (regresión simple)'
    else:
        raise ValueError("Debe proporcionar al menos un valor: cuareim_val o catalan_val.")

    return valor, (valor - n_std * std_used, valor + n_std * std_used), modelo






In [89]:
# Ejemplo de uso
print("--- Solo Cuareim ---")
v, (rmin, rmax), modelo = predecir_artigas(cuareim_val=11.05)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]\n")



--- Solo Cuareim ---
Modelo: Solo Cuareím Río (regresión simple)
Predicción: 9.36, Rango ±1std: [8.55, 10.16]



In [90]:
print("--- Solo Catalán ---")
v, (rmin, rmax), modelo = predecir_artigas(catalan_val=7.35)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]\n")



--- Solo Catalán ---
Modelo: Solo Catalán Grande (regresión simple)
Predicción: 8.69, Rango ±1std: [7.47, 9.92]



In [92]:
print("--- Cuareim + Catalán ---")
v, (rmin, rmax), modelo = predecir_artigas(cuareim_val=11.05, catalan_val=7.35)
print(f"Modelo: {modelo}\nPredicción: {v:.2f}, Rango ±1std: [{rmin:.2f}, {rmax:.2f}]")

--- Cuareim + Catalán ---
Modelo: Cuareím + 0.4·Catalán (regresión lineal)
Predicción: 9.21, Rango ±1std: [8.39, 10.03]
